Confiming the copy landed correctly — 3.44 GB, sitting in raw, matches the expected size for this file. So the data transfer is fully verified and done; no need to double-check that further.

In [10]:
from google.colab import drive
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Below we are checking: every key expected from the data dictionary structure showed up: all six categories (A, T, W, X_s, X_v, Y) split into _dev/_test, with _var name-list counterparts for everything except Y (which makes sense — Y is just the RUL target column, a single number per row, so there's no set of variable names to look up). No surprises or missing pieces here, which is a good sign the file is exactly what the proposal was built around.

In [11]:
import h5py
path = '/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/raw/N-CMAPSS_DS03-012.h5'
f = h5py.File(path, 'r')
print(list(f.keys()))

['A_dev', 'A_test', 'A_var', 'T_dev', 'T_test', 'T_var', 'W_dev', 'W_test', 'W_var', 'X_s_dev', 'X_s_test', 'X_s_var', 'X_v_dev', 'X_v_test', 'X_v_var', 'Y_dev', 'Y_test']


Next step: check the actual shapes and column counts against what docs/data-dictionary.md currently states.  

In [12]:
#Checking the actual shapes and columns counts
for key in f.keys():
    print(key, f[key].shape, f[key].dtype)

A_dev (5571277, 4) float64
A_test (4251560, 4) float64
A_var (4,) |S5
T_dev (5571277, 10) float64
T_test (4251560, 10) float64
T_var (10,) |S12
W_dev (5571277, 4) float64
W_test (4251560, 4) float64
W_var (4,) |S4
X_s_dev (5571277, 14) float64
X_s_test (4251560, 14) float64
X_s_var (14,) |S4
X_v_dev (5571277, 14) float64
X_v_test (4251560, 14) float64
X_v_var (14,) |S5
Y_dev (5571277, 1) int64
Y_test (4251560, 1) int64


Above

Above output - Everything matches. Every column count lines up exactly with what docs/data-dictionary.md assumes: W at 4 columns, X_s and X_v at 14 each, T at 10, A at 4, and Y at 1 (the RUL target, stored as int64 rather than float, which makes sense since it's a whole-number remaining-cycles count). No corrections needed to the data dictionary — it was written accurately.

The row counts are worth noting too: dev has 5,571,277 rows and test has 4,251,560, which adds up to 9,822,837 total — right in line with the proposal's "~9.8M records" figure. That's a second confirmation the file matches what the project was designed around.

Next: Decode the variable names, so you know exactly what each column represents rather than just a column index:

In [14]:
#ecode the variable names, so you know exactly what each column represents rather than just a column index:
w_names = [n.decode() if isinstance(n, bytes) else n for n in f['W_var'][:]]
xs_names = [n.decode() if isinstance(n, bytes) else n for n in f['X_s_var'][:]]
print("W columns:", w_names)
print("X_s columns:", xs_names)

W columns: ['alt', 'Mach', 'TRA', 'T2']
X_s columns: ['T24', 'T30', 'T48', 'T50', 'P15', 'P2', 'P21', 'P24', 'Ps30', 'P40', 'P50', 'Nf', 'Nc', 'Wf']


Above output:That's the full column mapping — 4 flight-condition variables (alt, Mach, TRA, T2) and 14 real sensor names (T24 through Wf — a mix of temperatures, pressures, and rotational speeds, matching standard turbofan sensor naming conventions). This is genuinely useful to have confirmed

Next: heck — unit and cycle counts. Run:

In [15]:
import numpy as np

a_dev = f['A_dev'][:]
units = np.unique(a_dev[:, 0])
print("Number of units:", len(units))
print("Unit IDs:", units)

Number of units: 9
Unit IDs: [1. 2. 3. 4. 5. 6. 7. 8. 9.]


Above results implication for my praxis proposal: This is an important finding — DS03 has 9 units, not the ~100 the proposal describes. That's not an error in your download or a mistake in your commands; it reflects how N-CMAPSS is actually structured. NASA split the full N-CMAPSS collection across eight separate files (DS01 through DS08), each covering a different subset of engine units, flight classes, and fault-mode combinations rather than every file containing the whole dataset. The "100 units, 7 failure modes" figure in your proposal almost certainly describes the entire N-CMAPSS collection across all eight DS0x files combined, not what's inside DS03 alone.

This is worth resolving properly rather than guessing, since it affects how you describe your dataset in the actual work. A couple of ways to pin it down:

The NASA example notebook you already have in your notebooks/ folder (N_CMAPSS_Example_data_loading_and_exploration.ipynb) likely documents exactly which units and fault modes are in DS03 specifically — worth opening it and checking, since it's the authoritative reference for this dataset.

Separately, it's worth checking the exact wording in your original proposal/pptx around "100 units, 7 failure modes" — if it was describing N-CMAPSS in general rather than committing specifically to DS03, then working with 9 units in DS03 is completely fine and expected; the docs just need a small correction to say "9 units from the DS03 subset" rather than implying all 100 units are available in this file.

Next: One more useful number before closing this out — how many flight cycles does each of these 9 units actually have? Run:

In [16]:
example_unit = units[0]
mask = a_dev[:, 0] == example_unit
print(f"Unit {example_unit} has {mask.sum()} rows (flight-cycle samples)")

Unit 1.0 has 308489 rows (flight-cycle samples)


Above result:That confirms it — 308,489 rows for unit 1 alone, and this is worth interpreting correctly: this is not 308,489 flight cycles. N-CMAPSS records sensor readings at a high sampling rate across each flight, so a single flight cycle spans many rows of time-series data. To get the actual cycle count, we need to know which column in A_dev holds the cycle number — right now we've been assuming column 0 is unit ID, but we haven't actually confirmed the column order.

Next steps:Let's decode A_var the same way you decoded W_var and X_s_var, so we know exactly what each of the 4 columns in A_dev means:

In [17]:
a_names = [n.decode() if isinstance(n, bytes) else n for n in f['A_var'][:]]
print("A columns:", a_names)

A columns: ['unit', 'cycle', 'Fc', 'hs']


Above result: Exactly as expected — unit, cycle, Fc (flight class), hs (health state). Now we can get the real cycle count using the correct column:

In [18]:
cycle_col = a_names.index('cycle')
cycles_unit1 = np.unique(a_dev[mask, cycle_col])
print(f"Unit {example_unit} has {len(cycles_unit1)} distinct flight cycles")
print("Flight classes present:", np.unique(a_dev[mask, a_names.index('Fc')]))
print("Health states present:", np.unique(a_dev[mask, a_names.index('hs')]))

Unit 1.0 has 72 distinct flight cycles
Flight classes present: [1.]
Health states present: [0. 1.]


Above Resultts: That's a clean, sensible result. Unit 1 flew 72 cycles, all within flight class 1 (N-CMAPSS groups flights into three length classes — 1 is typically the shortest), and its health state moved between 1 (healthy) and 0 (degraded) across its life — exactly the pattern you'd want for a fault-diagnosis dataset, since it means this unit's data actually spans both normal and degraded operation rather than being all-healthy or all-faulty.

On the "7 failure modes" figure from the proposal: hs here is just a binary healthy/unhealthy flag, not a specific fault-type label, so that number almost certainly refers to something at a higher level — most likely the different fault-mode combinations NASA built into the eight separate DS0x files (DS03 representing one particular combination), rather than 7 distinct labels living inside this one file. That's worth confirming against the NASA example notebook or the Chao et al. citation rather than guessing further from the data alone — not something to resolve right now, just flagging it as a small follow-up before you write it into your final docs.

Let's run the verification pass. Add these as new cells in your 01_eda.ipynb notebook (right after your existing DS03 checks) — same pattern, just pointed at the two new files.
Check DS01:

In [19]:
path_ds01 = '/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/raw/N-CMAPSS_DS01-005.h5'
f1 = h5py.File(path_ds01, 'r')
print("Keys:", list(f1.keys()))
for key in f1.keys():
    print(key, f1[key].shape, f1[key].dtype)

Keys: ['A_dev', 'A_test', 'A_var', 'T_dev', 'T_test', 'T_var', 'W_dev', 'W_test', 'W_var', 'X_s_dev', 'X_s_test', 'X_s_var', 'X_v_dev', 'X_v_test', 'X_v_var', 'Y_dev', 'Y_test']
A_dev (4906636, 4) float64
A_test (2735232, 4) float64
A_var (4,) |S5
T_dev (4906636, 10) float64
T_test (2735232, 10) float64
T_var (10,) |S12
W_dev (4906636, 4) float64
W_test (2735232, 4) float64
W_var (4,) |S4
X_s_dev (4906636, 14) float64
X_s_test (2735232, 14) float64
X_s_var (14,) |S4
X_v_dev (4906636, 14) float64
X_v_test (2735232, 14) float64
X_v_var (14,) |S5
Y_dev (4906636, 1) int64
Y_test (2735232, 1) int64


Results above: Clean match — every column count for DS01 lines up exactly with DS03: W=4, X_s=14, X_v=14, T=10, A=4, Y=1. Same key names too. That's exactly what preprocessing.py needs — all three files can be loaded and combined with identical logic since their structure is consistent.

Row counts: dev has 4,906,636 rows, test has 2,735,232 — a total of 7,641,868 for this file alone.

#Now run the DS08a check:


In [20]:
path_ds08a = '/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/raw/N-CMAPSS_DS08a-009.h5'
f8 = h5py.File(path_ds08a, 'r')
print("Keys:", list(f8.keys()))
for key in f8.keys():
    print(key, f8[key].shape, f8[key].dtype)

Keys: ['A_dev', 'A_test', 'A_var', 'T_dev', 'T_test', 'T_var', 'W_dev', 'W_test', 'W_var', 'X_s_dev', 'X_s_test', 'X_s_var', 'X_v_dev', 'X_v_test', 'X_v_var', 'Y_dev', 'Y_test']
A_dev (4885389, 4) float64
A_test (3722997, 4) float64
A_var (4,) |S5
T_dev (4885389, 10) float64
T_test (3722997, 10) float64
T_var (10,) |S12
W_dev (4885389, 4) float64
W_test (3722997, 4) float64
W_var (4,) |S4
X_s_dev (4885389, 14) float64
X_s_test (3722997, 14) float64
X_s_var (14,) |S4
X_v_dev (4885389, 14) float64
X_v_test (3722997, 14) float64
X_v_var (14,) |S5
Y_dev (4885389, 1) int64
Y_test (3722997, 1) int64


Results above: Another clean match — DS08a's columns are identical in structure to DS01 and DS03: W=4, X_s=14, X_v=14, T=10, A=4, Y=1, same key names throughout. All three files are now confirmed structurally consistent, which means preprocessing.py can load and combine them with one shared loading function rather than needing special-case logic per file.

Row counts across all three: DS01 (7,641,868) + DS03 (9,822,837) + DS08a (8,608,386) = 26,073,091 total rows once combined. That's a substantial training set for the MA1DCNN.

In [21]:
a1 = f1['A_dev'][:]
print("DS01 dev units:", len(np.unique(a1[:, 0])))

a8 = f8['A_dev'][:]
print("DS08a dev units:", len(np.unique(a8[:, 0])))

DS01 dev units: 6
DS08a dev units: 9


Building  one efficient cell that loops through all 7 new files and prints the same checks you did individually for DS01/DS03/DS08a — key structure, shapes, unit count, and cycle range. That's faster than writing 7 separate blocks.

All 6 files completed cleanly with no errors this time. Combined with your earlier checks on DS01, DS03, and DS08a, that's all 9 files in your dataset now fully verified: consistent column structure across every file (W=4, X_s=14, X_v=14, T=10, A=4, Y=1), correct unit lists, sensible cycle ranges, and both healthy and degraded (hs = 0/1) samples present in each one.

In [23]:
import h5py
import numpy as np

# The 7 files copied into Drive today, not yet verified
new_files = [
    "N-CMAPSS_DS02-006.h5",
    "N-CMAPSS_DS04.h5",
    "N-CMAPSS_DS05.h5",
    "N-CMAPSS_DS06.h5",
    "N-CMAPSS_DS07.h5",
    "N-CMAPSS_DS08c-008.h5",

]

base_path = "/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/raw/"

for fname in new_files:
    path = base_path + fname
    with h5py.File(path, "r") as f:  # open the file read-only
        print("=" * 60)
        print(fname)
        print("Keys:", list(f.keys()))  # should match every other file's key set

        # Column counts should match what was already verified for DS01/DS03/DS08a:
        # W=4, X_s=14, X_v=14, T=10, A=4, Y=1 — a mismatch here would mean this file's
        # structure is different and needs a closer look before combining datasets
        print("W_dev shape:", f["W_dev"].shape)
        print("X_s_dev shape:", f["X_s_dev"].shape)
        print("X_v_dev shape:", f["X_v_dev"].shape)
        print("T_dev shape:", f["T_dev"].shape)
        print("A_dev shape:", f["A_dev"].shape)
        print("Y_dev shape:", f["Y_dev"].shape)

        a_dev = f["A_dev"][:]  # load the auxiliary group: unit, cycle, Fc, hs (columns 0-3)

        units = np.unique(a_dev[:, 0])  # column 0 = engine unit ID
        print("Dev units:", units)

        print("Cycle range:", a_dev[:, 1].min(), "-", a_dev[:, 1].max())  # column 1 = cycle number

        # column 3 = health state (hs); should show both 0 (degraded) and 1 (healthy)
        # present in at least most files, confirming the labeling logic has data to work with
        print("hs values present:", np.unique(a_dev[:, 3]))

N-CMAPSS_DS02-006.h5
Keys: ['A_dev', 'A_test', 'A_var', 'T_dev', 'T_test', 'T_var', 'W_dev', 'W_test', 'W_var', 'X_s_dev', 'X_s_test', 'X_s_var', 'X_v_dev', 'X_v_test', 'X_v_var', 'Y_dev', 'Y_test']
W_dev shape: (5263447, 4)
X_s_dev shape: (5263447, 14)
X_v_dev shape: (5263447, 14)
T_dev shape: (5263447, 10)
A_dev shape: (5263447, 4)
Y_dev shape: (5263447, 1)
Dev units: [ 2.  5. 10. 16. 18. 20.]
Cycle range: 1.0 - 89.0
hs values present: [0. 1.]
N-CMAPSS_DS04.h5
Keys: ['A_dev', 'A_test', 'A_var', 'T_dev', 'T_test', 'T_var', 'W_dev', 'W_test', 'W_var', 'X_s_dev', 'X_s_test', 'X_s_var', 'X_v_dev', 'X_v_test', 'X_v_var', 'Y_dev', 'Y_test']
W_dev shape: (6377452, 4)
X_s_dev shape: (6377452, 14)
X_v_dev shape: (6377452, 14)
T_dev shape: (6377452, 10)
A_dev shape: (6377452, 4)
Y_dev shape: (6377452, 1)
Dev units: [1. 2. 3. 4. 5. 6.]
Cycle range: 1.0 - 100.0
hs values present: [0. 1.]
N-CMAPSS_DS05.h5
Keys: ['A_dev', 'A_test', 'A_var', 'T_dev', 'T_test', 'T_var', 'W_dev', 'W_test', 'W_var', '

In [24]:
# Compare source (Colab local) vs. Drive copy — sizes should match exactly
!ls -la ncmapss_extracted/data_set/data_set/N-CMAPSS_DS08d-010.h5
!ls -la "/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/raw/N-CMAPSS_DS08d-010.h5"

ls: cannot access 'ncmapss_extracted/data_set/data_set/N-CMAPSS_DS08d-010.h5': No such file or directory
ls: cannot access '/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/raw/N-CMAPSS_DS08d-010.h5': No such file or directory


Let's decode them using DS03, since that's the file you've explored most already. Add this cell in 01_eda.ipynb:

Both decoded cleanly, and they line up well with the documentation in the paper you uploaded earlier:

T_var (Model Health Parameters) matches Table 4 in that reference paper exactly — all 10: fan_eff_mod, fan_flow_mod, LPC_eff_mod, LPC_flow_mod, HPC_eff_mod, HPC_flow_mod, HPT_eff_mod, HPT_flow_mod, LPT_eff_mod, LPT_flow_mod.

X_v_var (Virtual Sensors) is a 14-column subset of the paper's Table 3, which lists 18 possible virtual sensors — your files include 14 of them (T40, P30, P45, W21, W22, W25, W31, W32, W48, W50, SmFan, SmLPC, SmHPC, phi), leaving out epr, NRf, NRc, and PCNfR. That's expected — not every N-CMAPSS release includes every possible virtual sensor column, and this confirms your files are internally consistent with what NASA documented.

That closes out the last EDA gap — every column group across all 9 files is now fully identified: W, X_s, X_v, T, A, and Y.

In [25]:
 import h5py

path = "/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/raw/N-CMAPSS_DS03-012.h5"

with h5py.File(path, "r") as f:
    t_var = f["T_var"][:]        # names for the T group (10 columns)
    x_v_var = f["X_v_var"][:]    # names for the X_v group (14 columns)

    # These are stored as byte strings (e.g. b'T24'), so decode to normal text
    t_names = [name.decode("utf-8") for name in t_var]
    x_v_names = [name.decode("utf-8") for name in x_v_var]

    print("T_var (Model Health Parameters, 10 cols):")
    print(t_names)
    print()
    print("X_v_var (Virtual Sensors, 14 cols):")
    print(x_v_names)

T_var (Model Health Parameters, 10 cols):
['fan_eff_mod', 'fan_flow_mod', 'LPC_eff_mod', 'LPC_flow_mod', 'HPC_eff_mod', 'HPC_flow_mod', 'HPT_eff_mod', 'HPT_flow_mod', 'LPT_eff_mod', 'LPT_flow_mod']

X_v_var (Virtual Sensors, 14 cols):
['T40', 'P30', 'P45', 'W21', 'W22', 'W25', 'W31', 'W32', 'W48', 'W50', 'SmFan', 'SmLPC', 'SmHPC', 'phi']


Below goal is:see "Files in use" table, though, for  added files (DS02, DS04, DS05, DS06, DS07, DS08c) — my earlier check only captured dev shapes, not test shapes.

In [26]:
 import h5py

files_needed = [
    "N-CMAPSS_DS02-006.h5",
    "N-CMAPSS_DS04.h5",
    "N-CMAPSS_DS05.h5",
    "N-CMAPSS_DS06.h5",
    "N-CMAPSS_DS07.h5",
    "N-CMAPSS_DS08c-008.h5",
]
base_path = "/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/raw/"

for fname in files_needed:
    path = base_path + fname
    with h5py.File(path, "r") as f:
        dev_rows = f["W_dev"].shape[0]
        test_rows = f["W_test"].shape[0]
        print(f"{fname}: dev={dev_rows:,}  test={test_rows:,}  total={dev_rows+test_rows:,}")

N-CMAPSS_DS02-006.h5: dev=5,263,447  test=1,253,743  total=6,517,190
N-CMAPSS_DS04.h5: dev=6,377,452  test=3,602,561  total=9,980,013
N-CMAPSS_DS05.h5: dev=4,350,606  test=2,562,046  total=6,912,652
N-CMAPSS_DS06.h5: dev=4,257,209  test=2,522,447  total=6,779,656
N-CMAPSS_DS07.h5: dev=4,350,176  test=2,869,786  total=7,219,962
N-CMAPSS_DS08c-008.h5: dev=4,299,918  test=2,117,819  total=6,417,737
